In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_validate,
    GridSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    root_mean_squared_error,
)

clf_df = pd.read_csv("data/classification_escalation.csv")
reg_df = pd.read_csv("data/resolution_time_regression.csv")

display(clf_df.head(), reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [3]:
import os
print("Current working directory:", os.getcwd())
print("Items here:", os.listdir("."))

Current working directory: C:\Users\pc\OneDrive\Desktop\Lab 4
Items here: ['.ipynb_checkpoints', 'data', 'lab4.ipynb']


In [5]:
print("=== CLASSIFICATION DATA (clf_df) ===")
print("Shape:", clf_df.shape)
print("\nMissing values per column:\n", clf_df.isna().sum())
print("\nData types:\n", clf_df.dtypes)
print("\nTarget 'escalated' counts:\n", clf_df["escalated"].value_counts(dropna=False))
print("\nTarget 'escalated' proportions:\n", clf_df["escalated"].value_counts(normalize=True))

print("\n" + "=" * 40 + "\n")

print("=== REGRESSION DATA (reg_df) ===")
print("Shape:", reg_df.shape)
print("\nMissing values per column:\n", reg_df.isna().sum())
print("\nData types:\n", reg_df.dtypes)
print("\nTarget 'resolution_time_hours' summary:\n", reg_df["resolution_time_hours"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

=== CLASSIFICATION DATA (clf_df) ===
Shape: (900, 7)

Missing values per column:
 ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64

Data types:
 ticket_id                     str
complexity_score          float64
team_experience_years     float64
priority                      str
channel                       str
customer_tenure_months    float64
escalated                   int64
dtype: object

Target 'escalated' counts:
 escalated
0    709
1    191
Name: count, dtype: int64

Target 'escalated' proportions:
 escalated
0    0.787778
1    0.212222
Name: proportion, dtype: float64


=== REGRESSION DATA (reg_df) ===
Shape: (800, 7)

Missing values per column:
 ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months 

In [8]:
X_clf = clf_df.drop(columns=["ticket_id", "escalated"])
y_clf = clf_df["escalated"]

In [9]:
print("X_clf shape:", X_clf.shape)
print("y_clf shape:", y_clf.shape)
print("Remaining feature columns:", list(X_clf.columns))
print("'ticket_id' in X_clf:", "ticket_id" in X_clf.columns)
print("'escalated' in X_clf:", "escalated" in X_clf.columns)

X_clf shape: (900, 5)
y_clf shape: (900,)
Remaining feature columns: ['complexity_score', 'team_experience_years', 'priority', 'channel', 'customer_tenure_months']
'ticket_id' in X_clf: False
'escalated' in X_clf: False


In [10]:
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    stratify=y_clf,
    random_state=42,
)

In [11]:
print("X_train_clf shape:", X_train_clf.shape, "| y_train_clf shape:", y_train_clf.shape)
print("X_test_clf shape:", X_test_clf.shape, "| y_test_clf shape:", y_test_clf.shape)
print("\nOriginal class proportions:\n", y_clf.value_counts(normalize=True))
print("\nTrain class proportions:\n", y_train_clf.value_counts(normalize=True))
print("\nTest class proportions:\n", y_test_clf.value_counts(normalize=True))

X_train_clf shape: (720, 5) | y_train_clf shape: (720,)
X_test_clf shape: (180, 5) | y_test_clf shape: (180,)

Original class proportions:
 escalated
0    0.787778
1    0.212222
Name: proportion, dtype: float64

Train class proportions:
 escalated
0    0.7875
1    0.2125
Name: proportion, dtype: float64

Test class proportions:
 escalated
0    0.788889
1    0.211111
Name: proportion, dtype: float64


In [12]:
numeric_cols_clf = [
    "complexity_score",
    "team_experience_years",
    "customer_tenure_months",
]
categorical_cols_clf = [
    "priority",
    "channel",
]

numeric_transformer_clf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer_clf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

clf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_clf, numeric_cols_clf),
        ("cat", categorical_transformer_clf, categorical_cols_clf),
    ]
)

In [13]:
X_train_clf_transformed = clf_preprocessor.fit_transform(X_train_clf)
print("Transformed X_train_clf shape:", X_train_clf_transformed.shape)
print("Any NaNs after preprocessing:", np.isnan(X_train_clf_transformed).sum())

Transformed X_train_clf shape: (720, 11)
Any NaNs after preprocessing: 0


In [14]:
numeric_cols_clf = [
    "complexity_score",
    "team_experience_years",
    "customer_tenure_months",
]
categorical_cols_clf = [
    "priority",
    "channel",
]

numeric_transformer_clf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer_clf = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

clf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_clf, numeric_cols_clf),
        ("cat", categorical_transformer_clf, categorical_cols_clf),
    ]
)

In [15]:
X_train_clf_prep = clf_preprocessor.fit_transform(X_train_clf)
print("Transformed X_train_clf shape:", X_train_clf_prep.shape)
print("Missing values after preprocessing:", np.isnan(X_train_clf_prep).sum())

Transformed X_train_clf shape: (720, 11)
Missing values after preprocessing: 0


In [16]:
clf_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "k-NN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

clf_cv_rows = []
for name, model in clf_models.items():
    pipe = Pipeline(
        steps=[
            ("prep", clf_preprocessor),
            ("model", model),
        ]
    )
    cv_res = cross_validate(
        pipe,
        X_train_clf,
        y_train_clf,
        cv=skf,
        scoring="f1",
    )
    scores = cv_res["test_score"]
    clf_cv_rows.append(
        {
            "Model": name,
            "F1 Mean": scores.mean(),
            "F1 Std": scores.std(),
            "Fold F1 Scores": np.round(scores, 4),
        }
    )

clf_cv_summary = pd.DataFrame(clf_cv_rows)
display(clf_cv_summary)

,Model,F1 Mean,F1 Std,Fold F1 Scores
0,Logistic Regression,0.392651,0.062747,"[0.3913, 0.3256, 0.3636, 0.3721, 0.5106]"
1,k-NN,0.360835,0.050295,"[0.3673, 0.3256, 0.4255, 0.2857, 0.4]"
2,Decision Tree,0.330666,0.058126,"[0.4, 0.2295, 0.3226, 0.3729, 0.3284]"


In [17]:
tune_pipe = Pipeline(
    steps=[
        ("prep", clf_preprocessor),
        ("model", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

param_grid_clf = {
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__class_weight": [None, "balanced"],
}

grid_clf = GridSearchCV(
    estimator=tune_pipe,
    param_grid=param_grid_clf,
    cv=skf,
    scoring="f1",
    n_jobs=-1,
)

grid_clf.fit(X_train_clf, y_train_clf)

print("Best F1 CV score:", grid_clf.best_score_)
print("Best parameters:", grid_clf.best_params_)

Best F1 CV score: 0.5199932976178999
Best parameters: {'model__C': 0.1, 'model__class_weight': 'balanced'}


In [18]:
best_clf_model = grid_clf.best_estimator_
y_test_pred_clf = best_clf_model.predict(X_test_clf)

cm = confusion_matrix(y_test_clf, y_test_pred_clf)
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:\n", cm)
print(f"\nTrue Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")
print("\nClassification Report:\n", classification_report(y_test_clf, y_test_pred_clf))

Confusion Matrix:
 [[117  25]
 [ 11  27]]

True Negatives (TN): 117
False Positives (FP): 25
False Negatives (FN): 11
True Positives (TP): 27

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.82      0.87       142
           1       0.52      0.71      0.60        38

    accuracy                           0.80       180
   macro avg       0.72      0.77      0.73       180
weighted avg       0.83      0.80      0.81       180



In [19]:
X_reg = reg_df.drop(columns=["ticket_id", "resolution_time_hours"])
y_reg = reg_df["resolution_time_hours"]

In [20]:
print("X_reg shape:", X_reg.shape)
print("y_reg shape:", y_reg.shape)
print("Remaining feature columns:", list(X_reg.columns))
print("'ticket_id' in X_reg:", "ticket_id" in X_reg.columns)
print("'resolution_time_hours' in X_reg:", "resolution_time_hours" in X_reg.columns)

X_reg shape: (800, 5)
y_reg shape: (800,)
Remaining feature columns: ['complexity_score', 'team_experience_years', 'priority', 'channel', 'customer_tenure_months']
'ticket_id' in X_reg: False
'resolution_time_hours' in X_reg: False


In [21]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42,
)

In [22]:
print("X_train_reg shape:", X_train_reg.shape, "| y_train_reg shape:", y_train_reg.shape)
print("X_test_reg shape:", X_test_reg.shape, "| y_test_reg shape:", y_test_reg.shape)
print("Train target mean / median:", round(y_train_reg.mean(), 2), "/", round(y_train_reg.median(), 2))
print("Test target mean / median:", round(y_test_reg.mean(), 2), "/", round(y_test_reg.median(), 2))

X_train_reg shape: (640, 5) | y_train_reg shape: (640,)
X_test_reg shape: (160, 5) | y_test_reg shape: (160,)
Train target mean / median: 5.77 / 5.34
Test target mean / median: 6.3 / 5.63


In [23]:
numeric_cols_reg = [
    "complexity_score",
    "team_experience_years",
    "customer_tenure_months",
]
categorical_cols_reg = [
    "priority",
    "channel",
]

numeric_transformer_reg = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer_reg = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

reg_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_reg, numeric_cols_reg),
        ("cat", categorical_transformer_reg, categorical_cols_reg),
    ]
)

In [24]:
X_train_reg_prep = reg_preprocessor.fit_transform(X_train_reg)
print("Transformed X_train_reg shape:", X_train_reg_prep.shape)
print("Missing values after preprocessing:", np.isnan(X_train_reg_prep).sum())

Transformed X_train_reg shape: (640, 11)
Missing values after preprocessing: 0


In [25]:
reg_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(random_state=42),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

reg_cv_rows = []
for name, model in reg_models.items():
    pipe = Pipeline(
        steps=[
            ("prep", reg_preprocessor),
            ("model", model),
        ]
    )
    cv_res = cross_validate(
        pipe,
        X_train_reg,
        y_train_reg,
        cv=kf,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
        },
    )
    mae_scores = -cv_res["test_mae"]
    rmse_scores = -cv_res["test_rmse"]
    reg_cv_rows.append(
        {
            "Model": name,
            "MAE Mean": mae_scores.mean(),
            "MAE Std": mae_scores.std(),
            "RMSE Mean": rmse_scores.mean(),
            "RMSE Std": rmse_scores.std(),
        }
    )

reg_cv_summary = pd.DataFrame(reg_cv_rows)
display(reg_cv_summary)

,Model,MAE Mean,MAE Std,RMSE Mean,RMSE Std
0,Linear Regression,2.035134,0.171650,2.960627,0.584458
1,Ridge,2.035127,0.171404,2.960478,0.583462
2,Decision Tree,3.239097,0.053807,4.587631,0.374918


In [26]:
ridge_tune_pipe = Pipeline(
    steps=[
        ("prep", reg_preprocessor),
        ("model", Ridge(random_state=42)),
    ]
)

param_grid_ridge = {
    "model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 500.0],
}

grid_ridge = GridSearchCV(
    estimator=ridge_tune_pipe,
    param_grid=param_grid_ridge,
    cv=kf,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)

grid_ridge.fit(X_train_reg, y_train_reg)

print("Best CV MAE (hours):", -grid_ridge.best_score_)
print("Best Ridge parameters:", grid_ridge.best_params_)

Best CV MAE (hours): 2.035118085555478
Best Ridge parameters: {'model__alpha': 0.1}


In [28]:
best_reg_model = grid_ridge.best_estimator_
y_test_pred_reg = best_reg_model.predict(X_test_reg)

test_mae = mean_absolute_error(y_test_reg, y_test_pred_reg)
test_rmse = root_mean_squared_error(y_test_reg, y_test_pred_reg)

residuals = y_test_reg - y_test_pred_reg
abs_errors = np.abs(residuals)

print(f"Test MAE (hours):  {test_mae:.4f}")
print(f"Test RMSE (hours): {test_rmse:.4f}")
print("\nResidual summary (Actual - Predicted):")
print(residuals.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))
print("\nTop 5 largest absolute errors:")
error_check = pd.DataFrame(
    {
        "Actual": y_test_reg,
        "Predicted": np.round(y_test_pred_reg, 2),
        "Residual": np.round(residuals, 2),
        "Abs_Error": np.round(abs_errors, 2),
    }
).sort_values(by="Abs_Error", ascending=False)
display(error_check.head(5))

Test MAE (hours):  2.2301
Test RMSE (hours): 3.4028

Residual summary (Actual - Predicted):
count    160.000000
mean       0.288741
std        3.401129
min       -7.140522
5%        -3.842931
25%       -1.654386
50%       -0.213825
75%        1.693768
95%        4.382932
max       21.109448
Name: resolution_time_hours, dtype: float64

Top 5 largest absolute errors:


,Actual,Predicted,Residual,Abs_Error
398,27.488229,6.38,21.11,21.11
49,24.108510,4.84,19.27,19.27
537,15.582888,3.36,12.22,12.22
490,5.110015,12.25,-7.14,7.14
306,5.311855,10.81,-5.50,5.50


# Lab 4 Report: Support Ticket Escalation & Resolution Time Modeling

## 1. Executive Summary
This report presents the design, cross-validation, hyperparameter tuning, and held-out test evaluation for two machine learning experiments built on support ticket data (excluding `ticket_id` as a non-predictive identifier and using `complexity_score`, `team_experience_years`, `customer_tenure_months`, `priority`, and `channel` as input features):

* **Experiment A (Binary Classification — Target: `escalated`):** A tuned `LogisticRegression` pipeline (`C=0.1`, `class_weight="balanced"`) achieved a **Test F1-score of 0.60** (with **0.71 recall**, **0.52 precision**, **25 false positives**, and **11 false negatives** across 180 test tickets), improving from a ~0.393 untuned 5-fold CV F1-score to ~0.520 CV F1-score.
* **Experiment B (Regression — Target: `resolution_time_hours`):** A tuned `Ridge` pipeline (`alpha=0.1`) achieved a **Test MAE of 2.2301 hours** and a **Test RMSE of 3.4028 hours** across 160 test tickets (closely matching 5-fold CV performance of ~2.035 hours MAE and ~2.960 hours RMSE), outperforming an unpruned `DecisionTreeRegressor` (~3.239 hours CV MAE).

---

## 2. Data Preparation & Leakage Prevention

### Methodology
* **Train/Test Split First:** Each experiment split data into an 80% training set and a 20% held-out test set (`test_size=0.2`, `random_state=42`) before any preprocessing or model selection, keeping test data untouched until final evaluation.
* **Leakage-Safe `ColumnTransformer` Inside `Pipeline`:**
  * Numeric features (`complexity_score`, `team_experience_years`, `customer_tenure_months`) used `SimpleImputer(strategy="median")` followed by `StandardScaler()`.
  * Categorical features (`priority`, `channel`) used `SimpleImputer(strategy="most_frequent")` followed by `OneHotEncoder(handle_unknown="ignore")`, expanding the 5 raw features into 11 transformed features.
  * Placing preprocessing and estimators inside a `Pipeline` ensured imputers, scalers, and encoders were fit strictly on training folds during `cross_validate` and `GridSearchCV`.

### Justification
* **Preventing Data Leakage:** Fitting scalers, imputers, or encoders on the full dataset or test set leaks information across splits and inflates validation/test scores. Keeping all steps inside `Pipeline` preserves honest generalization metrics.
* **Stratification in Classification Only:** Because `escalated` is class-imbalanced (~21.2% positive vs. ~78.8% negative), `stratify=y_clf` preserved class proportions across train and test sets. Because `resolution_time_hours` is continuous in regression, a standard split was used instead.
* **Median & Most-Frequent Imputation:** Median imputation resists skew/outliers in numeric ticket features, while most-frequent imputation and `handle_unknown="ignore"` handle missing or unseen categorical values safely.

---

## 3. Experiment A: Escalation Prediction (Classification)

### Model Selection & Tuning
* Evaluated across 5-fold stratified cross-validation on the training set, with `LogisticRegression` tuned via `GridSearchCV` using `scoring="f1"`.
* **Selected Configuration:** `LogisticRegression(C=0.1, class_weight="balanced")`, achieving a best 5-fold CV F1-score of **~0.5200** (up from ~0.3927 baseline).

### Held-Out Test Evaluation (180 Tickets)
| Metric / Outcome | Value | Interpretation |
| :--- | :--- | :--- |
| **Test F1-Score (Class 1)** | **0.60** | Harmonic balance of precision and recall on escalated tickets |
| **Test Recall (Class 1)** | **0.71** | Caught 27 of 38 actual escalations |
| **Test Precision (Class 1)** | **0.52** | 27 of 52 predicted escalations were truly escalated |
| **Test Accuracy** | **0.80** | Overall accuracy across all 180 test tickets |
| **Confusion Matrix Counts** | **TN: 117, FP: 25, FN: 11, TP: 27** | 25 false alarms vs. 11 missed escalations |

### Justification
* **Why F1 Instead of Accuracy:** Because ~79% of tickets are not escalated, predicting class `0` most of the time would yield deceptively high accuracy while missing urgent escalations. `F1` focuses directly on the minority positive class (`escalated = 1`).
* **Why Tuned `LogisticRegression` (`C=0.1`, `class_weight="balanced"`):**
  * `class_weight="balanced"` increases the penalty for misclassifying rarer `escalated = 1` tickets, boosting test recall to **0.71** and limiting missed escalations to **11** false negatives.
  * Regularization (`C=0.1`) keeps the linear decision boundary smooth and less prone to overfitting than high-variance unpruned tree splits on these 11 transformed features.
  * The trade-off is **25 false positives** (extra alerts/reviews on non-escalated tickets), which is acceptable when missing a true escalation carries higher risk.

---

## 4. Experiment B: Resolution Time Prediction (Regression)

### Model Selection & Tuning
* Evaluated across 5-fold `KFold` cross-validation on the 640-row training set:
  * **Linear Regression:** MAE Mean = `2.0351`h (Std = `0.1717`), RMSE Mean = `2.9606`h (Std = `0.5845`)
  * **Ridge:** MAE Mean = `2.0351`h (Std = `0.1714`), RMSE Mean = `2.9605`h (Std = `0.5835`)
  * **Decision Tree:** MAE Mean = `3.2391`h (Std = `0.0538`), RMSE Mean = `4.5876`h (Std = `0.3749`)
* **Selected Configuration:** `Ridge(alpha=0.1)` tuned via `GridSearchCV` using `"neg_mean_absolute_error"`, achieving a best CV MAE of **~2.0351 hours**.

### Held-Out Test Evaluation (160 Tickets)
| Metric | Value | Notes |
| :--- | :--- | :--- |
| **Test MAE** | **2.2301 hours** | Typical absolute error is ~2.23 hours |
| **Test RMSE** | **3.4028 hours** | Higher than MAE due to squared error penalty on outliers |
| **Residual Median (50%)** | **-0.2138 hours** | Near-zero median bias |
| **Residual IQR (25%–75%)** | **-1.6544 to +1.6938 hours** | Central 50% of errors span roughly ±1.7 hours |
| **Largest Positive Residuals** | **+21.11h** (row 398), **+19.27h** (row 49), **+12.22h** (row 537) | Long-tail resolution times far above predictions |

### Justification
* **Why `Ridge` (`alpha=0.1`) Over Unpruned `DecisionTreeRegressor`:** Linear models (`LinearRegression` and `Ridge`) generalize smoothly across the feature space without memorizing noisy local splits, reducing CV MAE from ~`3.24`h (unpruned tree) to ~`2.04`h. Mild L2 regularization (`alpha=0.1`) stabilizes coefficients without hurting fit.
* **Why RMSE Exceeds MAE (Right-Skewed Outliers):** `MAE` weights every hour of error linearly, capturing typical resolution error (~`2.23`h on test). `RMSE` squares residuals before taking the square root, so rare extreme tickets—such as row `398` (`27.49`h actual vs. `6.38`h predicted, `+21.11`h residual)—pull `RMSE` up to `3.40`h. Reporting both gives typical accuracy and extreme-error awareness.

---

## 5. Cross-Experiment Takeaways & Limitations
* **Linear Models vs. Unpruned Trees:** Across both tasks on these 5 ticket features, regularized linear models (`LogisticRegression` and `Ridge`) outperformed unpruned decision trees in generalization.
* **Potential Improvements:** For regression, long-tail resolution times motivate exploring target transformations (such as log-transforming skewed targets), regularized/pruned tree ensembles, or richer operational features (such as assignee workload, ticket age, or reassignment count) beyond the baseline features.